# Parkinson's Disease Classifier — nested, speaker-independent validation

Data: PC-GITA, 100 participants (50 PD, 50 controls), 6 recordings each (3 × /a/, 3 × /e/), both vowels pooled.

# Library Imports

In [ ]:
import os
import textwrap
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import StratifiedGroupKFold, LeaveOneGroupOut, GridSearchCV
from sklearn.metrics import balanced_accuracy_score, recall_score, roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
from sklearn.svm import SVC
warnings.filterwarnings('ignore')

# Data Loading

In [ ]:
if not os.path.exists('voice_waveshape'):
  !git clone https://github.com/joaquinr-uner/voice_waveshape.git

In [ ]:
CSV_PATH = 'voice_waveshape/Features_Parkinson.csv'
df = pd.read_csv(CSV_PATH)

Y = (df['Label'] == 'Parkinson').astype(int).to_numpy()   # 1 = Parkinson, 0 = Healthy (control)
G = df['PatientID'].to_numpy()
FEATURES = [c for c in df.columns if c not in ('PatientID', 'Label')]   # all 32 waveshape features
X = df[FEATURES].to_numpy()

spk = df.groupby('PatientID')
assert (spk['Label'].nunique() == 1).all()
print(len(FEATURES), 'features,', spk.ngroups, 'participants,', len(df), 'recordings')
spk['Label'].first().value_counts()

# Settings

Hyperparameter grids (as in the original notebook):
* SVM RBF: $C \in [10^{-1}, 10^{5}]$, $L = 13$; $\gamma \in [\frac{0.001}{N_{features}}, \frac{1000}{N_{features}}]$, $L = 13$
* Random Forest: number of trees $n_c$ and minimum samples per split $n_{samp}$
* AdaBoost: number of estimators $n_c$

Inner tuning criterion: recording-level balanced accuracy.

In [ ]:
N_OUTER = 10      # outer folds, 10-fold CV
N_INNER = 5       # inner folds (tuning)
SEED = 0
N_JOBS = -1


def make_models(n_features):
  return {
    'SVM RBF': (Pipeline([('sc', StandardScaler()), ('clf', SVC(class_weight='balanced'))]),
                {'clf__C': np.logspace(-1, 5, num=13),
                 'clf__gamma': 1/n_features*np.logspace(-3, 3, num=13)}),
    'Random Forest': (Pipeline([('sc', StandardScaler()), ('clf', RandomForestClassifier(random_state=SEED))]),
                      {'clf__n_estimators': [10, 25, 50, 75, 100, 150, 200, 300, 500],
                       'clf__min_samples_split': [2, 5, 10, 20, 30, 40, 50]}),
    'AdaBoost': (Pipeline([('sc', StandardScaler()), ('clf', AdaBoostClassifier(random_state=SEED))]),
                 {'clf__n_estimators': [10, 25, 50, 75, 100, 150, 200]}),
  }

MODELS = list(make_models(1))

RESULTS_DIR = 'results/PD'   # CSV tables, predictions and the PDF table are written here
os.makedirs(RESULTS_DIR, exist_ok=True)

# Nested CV and evaluation functions

In [ ]:
UNITS = ['Recording', 'Participant']
COLUMNS = ['Accuracy', 'Accuracy SD', 'Sensitivity', 'Specificity', 'AUC']


def continuous_score(model, X):
  # Positive = Parkinson; threshold at 0
  if hasattr(model, 'decision_function'):
    return model.decision_function(X)
  return model.predict_proba(X)[:, 1] - 0.5


def nested_cv(X, y, g, outer, pipe, grid, seed=SEED):
  # Out-of-fold predictions from a nested CV; hyperparameters tuned on outer-training participants only
  pred = np.full(len(y), -1)
  score = np.full(len(y), np.nan)
  fold = np.full(len(y), -1)
  chosen = []
  for k, (tr, te) in enumerate(outer.split(X, y, g)):
    assert not set(g[tr]) & set(g[te]), 'participant in both training and test'
    inner = StratifiedGroupKFold(n_splits=N_INNER, shuffle=True, random_state=seed)
    gs = GridSearchCV(pipe, grid, scoring='balanced_accuracy', cv=inner, n_jobs=N_JOBS)
    gs.fit(X[tr], y[tr], groups=g[tr])
    pred[te] = gs.predict(X[te])
    score[te] = continuous_score(gs.best_estimator_, X[te])
    fold[te] = k
    chosen.append({p.replace('clf__', ''): v for p, v in gs.best_params_.items()})
  return {'pred': pred, 'score': score, 'fold': fold, 'chosen': pd.DataFrame(chosen)}


def metrics(y, pred, score):
  return {'Accuracy': balanced_accuracy_score(y, pred),
          'Sensitivity': recall_score(y, pred, pos_label=1),
          'Specificity': recall_score(y, pred, pos_label=0),
          'AUC': roc_auc_score(y, score)}


def subject_decisions(out, y, g):
  # Soft voting: mean continuous score over the participant's recordings, PD if > 0
  d = pd.DataFrame({'spk': g, 'y': y, 'score': out['score'], 'fold': out['fold']})
  s = d.groupby('spk').agg(y=('y', 'first'), score=('score', 'mean'), fold=('fold', 'first'))
  s['pred'] = (s['score'] > 0).astype(int)
  return s


def evaluate(out, y, g, per_fold_sd):
  # Metrics per evaluation unit. per_fold_sd (10-fold): accuracy = mean ± sd over the outer folds;
  # otherwise (LOSO, one participant per outer fold) accuracy over all units, no sd.
  s = subject_decisions(out, y, g)
  units = {'Recording': (y, out['pred'], out['score'], out['fold']),
           'Participant': (s['y'].to_numpy(), s['pred'].to_numpy(), s['score'].to_numpy(), s['fold'].to_numpy())}
  by_unit = {}
  for unit, (yu, pu, su, fu) in units.items():
    m = metrics(yu, pu, su)
    m['Accuracy SD'] = np.nan
    if per_fold_sd:
      acc = [balanced_accuracy_score(yu[fu == k], pu[fu == k]) for k in np.unique(fu)]
      m['Accuracy'], m['Accuracy SD'] = np.mean(acc), np.std(acc)
    by_unit[unit] = m
  return s, by_unit


def results_table(protocol, by_clf):
  rows = [{'Protocol': protocol, 'Unit': u, 'Classifier': name, **by_clf[name][u]} for u in UNITS for name in by_clf]
  return pd.DataFrame(rows).set_index(['Protocol', 'Unit', 'Classifier'])[COLUMNS]

# Nested 10-fold CV
Outer `StratifiedGroupKFold` (10 folds, grouped by participant), inner participant-grouped 5-fold tuning.

In [ ]:
kfold_res = {}
for name, (pipe, grid) in make_models(len(FEATURES)).items():
  outer = StratifiedGroupKFold(n_splits=N_OUTER, shuffle=True, random_state=SEED)
  kfold_res[name] = nested_cv(X, Y, G, outer, pipe, grid, seed=SEED)
  print(f'done: {name}')

In [ ]:
kfold_subj, kfold_units = {}, {}
for name, out in kfold_res.items():
  kfold_subj[name], kfold_units[name] = evaluate(out, Y, G, per_fold_sd=True)
kfold_table = results_table('10-fold', kfold_units)
kfold_table.round(3)

# Nested Leave-One-Subject-Out CV
100 outer folds, one participant (6 recordings) each; the model is tuned and trained on the other 99 participants.

In [ ]:
loso_res = {}
for name, (pipe, grid) in make_models(len(FEATURES)).items():
  loso_res[name] = nested_cv(X, Y, G, LeaveOneGroupOut(), pipe, grid, seed=SEED)
  print(f'done: {name}')

In [ ]:
loso_subj, loso_units = {}, {}
for name, out in loso_res.items():
  loso_subj[name], loso_units[name] = evaluate(out, Y, G, per_fold_sd=False)
loso_table = results_table('LOSO', loso_units)
loso_table.round(3)

# Results table 6

In [1]:
results = pd.concat([kfold_table, loso_table])
results.to_csv(os.path.join(RESULTS_DIR, 'results_table.csv'), encoding='utf-8-sig')

BLOCKS = {('10-fold', 'Recording'): 'Nested 10-fold CV, recording level (n = 600 recordings)',
          ('10-fold', 'Participant'): 'Nested 10-fold CV, participant level (n = 100 participants, soft vote)',
          ('LOSO', 'Recording'): 'Nested leave-one-subject-out CV, recording level (n = 600 recordings)',
          ('LOSO', 'Participant'): 'Nested leave-one-subject-out CV, participant level (n = 100 participants, soft vote)'}
NOTES = [
  'Accuracy is the balanced accuracy (mean of sensitivity and specificity). Sensitivity: PD recordings or participants '
  'classified as PD; specificity: the same for controls.',
  'Nested 10-fold CV: accuracy is the mean ± SD over the 10 outer folds. Nested LOSO: each outer fold holds a single '
  'participant, so no SD is given. Sensitivity, specificity and AUC are computed on the pooled out-of-fold predictions.',
  "Recording level: each held-out recording is classified on its own. Participant level: the decision scores of a "
  "participant's 6 held-out recordings are averaged and the participant is labelled PD if the mean is > 0 (soft vote).",
  'AUC uses the continuous decision score (SVM and AdaBoost decision function; Random Forest probability − 0.5; at '
  'participant level, its mean over the 6 recordings).',
]
INK, INK2 = '#111111', '#555555'
COL_X = {'clf': 0.07, 'acc': 0.44, 'sens': 0.62, 'spec': 0.77, 'auc': 0.91}

fig = plt.figure(figsize=(8.5, 11))
rule = lambda yy, lw: fig.add_artist(plt.Line2D([0.07, 0.95], [yy, yy], color=INK, lw=lw))
y = 0.94
fig.text(0.07, y, "Parkinson's disease detection: nested cross-validation results", fontsize=14, weight='bold', color=INK)
y -= 0.022
fig.text(0.07, y, 'PC-GITA, 50 PD and 50 control participants, 6 recordings each; all 32 waveshape features.',
         fontsize=8.5, color=INK2)
y -= 0.04
for (protocol, unit), title in BLOCKS.items():
  with_sd = protocol == '10-fold'
  fig.text(0.07, y, title, fontsize=10, weight='bold', color=INK)
  y -= 0.012
  rule(y, 1.2)
  y -= 0.018
  head = {'clf': 'Classifier', 'acc': 'Accuracy (± SD)' if with_sd else 'Accuracy',
          'sens': 'Sensitivity', 'spec': 'Specificity', 'auc': 'AUC'}
  for k, t in head.items():
    fig.text(COL_X[k], y, t, fontsize=9, weight='bold', color=INK, ha='left' if k == 'clf' else 'center')
  y -= 0.010
  rule(y, 0.8)
  y -= 0.020
  for name, r in results.loc[(protocol, unit)].iterrows():
    acc = f"{r['Accuracy']:.3f} ± {r['Accuracy SD']:.3f}" if with_sd else f"{r['Accuracy']:.3f}"
    vals = {'clf': name, 'acc': acc, 'sens': f"{r['Sensitivity']:.3f}", 'spec': f"{r['Specificity']:.3f}", 'auc': f"{r['AUC']:.3f}"}
    for k, t in vals.items():
      fig.text(COL_X[k], y, t, fontsize=9, color=INK, ha='left' if k == 'clf' else 'center')
    y -= 0.020
  y += 0.008
  rule(y, 1.2)
  y -= 0.045
for n in NOTES:
  for line in textwrap.wrap(n, 128):
    fig.text(0.07, y, line, fontsize=7.5, color=INK2)
    y -= 0.014
  y -= 0.004
fig.savefig(os.path.join(RESULTS_DIR, 'results_table.pdf'))
plt.close(fig)
results.round(3)

NameError: name 'pd' is not defined

# Save results
Writes the out-of-fold predictions, the participant decisions and the hyperparameters selected in each outer fold, plus a pickle of the result objects.

In [ ]:
META = ['Protocol', 'Classifier']
hp, oof, part = [], [], []
for protocol, res, subj in (('10-fold', kfold_res, kfold_subj), ('LOSO', loso_res, loso_subj)):
  for name, out in res.items():
    tag = {'Protocol': protocol, 'Classifier': name}
    hp.append(out['chosen'].assign(**{'Outer fold': np.arange(1, len(out['chosen']) + 1)}, **tag))
    oof.append(pd.DataFrame({**tag, 'PatientID': G, 'PD': Y, 'Outer fold': out['fold'] + 1,
                             'Pred': out['pred'], 'Score': out['score']}))
    part.append(subj[name].assign(fold=subj[name]['fold'] + 1).rename_axis('PatientID').reset_index()
                .rename(columns={'y': 'PD', 'score': 'Mean score', 'fold': 'Outer fold', 'pred': 'Pred PD (soft vote)'})
                .assign(**tag))

def _save(frames, name):
  d = pd.concat(frames, ignore_index=True)
  d[META + [c for c in d.columns if c not in META]].to_csv(os.path.join(RESULTS_DIR, f'{name}.csv'), index=False)

_save(hp, 'hyperparameters_per_outer_fold')
_save(oof, 'out_of_fold_predictions_recordings')
_save(part, 'participant_decisions')
pd.to_pickle({'kfold_res': kfold_res, 'loso_res': loso_res, 'kfold_subj': kfold_subj, 'loso_subj': loso_subj,
              'results': results}, os.path.join(RESULTS_DIR, 'results.pkl'))
print('saved to', RESULTS_DIR, sorted(os.listdir(RESULTS_DIR)))